# RTENSO2026: sensitivity to the land, sea-ice and river initial conditions

RTENSO2026 `clim` and `obs` share the ERA5 atmosphere and CFSR ocean ICs and differ only
in the land, sea-ice and river restarts (a BruteForce year closest to the same-date
climatology vs closest to the ERA5 anomaly). For the May and Nov 2015 starts, the
Reanalysis (BruteForce) 2015 hindcasts use the actual 2015 restarts.

This notebook compares the raw ensembles (no anomalies needed: same model, dates and
leads) and asks how large the IC differences are and how fast they fade:

* land-mean difference by lead with a member-based 95% interval (Welch t test, 10 vs 10 members);
* the fraction of land with a significant difference by lead (IC memory);
* difference maps at selected leads, stippled where p < 0.05;
* hemispheric sea-ice area against HadISST2.

Land and sea-ice fields need the RTENSO2026 `lnd_mon1` and `ice_mon1` post-processing;
cases that are not processed yet are skipped.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from matplotlib.colors import BoundaryNorm
from cartopy.util import add_cyclic_point
from esp_lab.utils import colormap_utils as mycolors
from esp_lab.utils.netcdf_utils import atomic_to_netcdf

import rtenso_case as rc

def to_datetime(times):
    return [datetime(t.year, t.month, 15) for t in np.atleast_1d(times)]

### Run controls

In [ ]:
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"         # Feb/Aug drift correction: "lead_drift" or "target_month"

FIELDS = ["TREFHT", "TS", "PRECT", "TSA", "SOILWATER_10CM", "H2OSNO", "TWS"]
MAP_FIELD = "SOILWATER_10CM"     # field for the difference maps
PAIRS = [("RTENSO2026-clim", "RTENSO2026-obs"),
         ("RTENSO2026-clim", "Reanalysis-2015"),
         ("RTENSO2026-obs", "Reanalysis-2015")]
INIT_MONTHS = [2, 5, 8, 11]
MONTH_NAMES = {2: "Feb", 5: "May", 8: "Aug", 11: "Nov"}

# Hindcast year and cases. Each case reads
# <raw_model_root>/<case_prefix>_<CASE_YEAR><MM>0100/ENxx/post/...; "source" names
# the cached files, "init_months" the starts the case has.
CASE_YEAR = 2015
MODEL_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5"
CASES = {
    "RTENSO2026-clim": {
        "case_prefix": f"{MODEL_PREFIX}_RTENSO2026_clim",
        "source": "RTENSO2026_clim",
        "init_months": (2, 5, 8, 11),
        "color": "tab:blue",
    },
    "RTENSO2026-obs": {
        "case_prefix": f"{MODEL_PREFIX}_RTENSO2026_obs",
        "source": "RTENSO2026_obs",
        "init_months": (2, 5, 8, 11),
        "color": "tab:orange",
    },
    "Reanalysis-2015": {
        "case_prefix": f"{MODEL_PREFIX}_BruteForce",
        "source": "Reanalysis",
        "init_months": (5, 11),
        "color": "tab:green",
    },
}
MAP_LEADS = [0, 2, 5, 11]
ALPHA = 0.05
LAND_THRESHOLD = 0.5             # EAM LANDFRAC for land means

OUTDIR = rc.diag_dir("RTENSO2026", "case_study", "ic_sensitivity")
OUTDIR.mkdir(parents=True, exist_ok=True)
land = rc.land_fraction() >= LAND_THRESHOLD

### Land-mean members and pair statistics

Only land means (and the ensemble mean and p value for `MAP_FIELD`) are kept, to limit memory.

In [ ]:
%%time
land_mean = {}     # (case, init, field) -> (member, lead)
maps = {}          # (pair, init) -> Dataset(diff, p) for MAP_FIELD
sig_fraction = {}  # (pair, init, field) -> (lead)

for field in FIELDS:
    ens = {}
    for case in CASES:
        for init in INIT_MONTHS:
            if init not in CASES[case]["init_months"]:
                continue
            da = rc.open_hindcast(CASES[case]["case_prefix"], CASE_YEAR, init, field)
            if da is None:
                continue
            ens[(case, init)] = da.where(land)
            land_mean[(case, init, field)] = rc.box_mean(ens[(case, init)], [0.0, 360.0, -90.0, 90.0]).load()
    for a, b in PAIRS:
        for init in INIT_MONTHS:
            if (a, init) not in ens or (b, init) not in ens:
                continue
            _, p = rc.welch_t(ens[(a, init)], ens[(b, init)])
            sig = (p < ALPHA).where(ens[(a, init)].isel(member=0).notnull())
            w = rc.area_weights(sig["lat"])
            sig_fraction[((a, b), init, field)] = sig.weighted(w).mean(("lat", "lon")).load()
            if field == MAP_FIELD:
                maps[((a, b), init)] = xr.Dataset({
                    "diff": ens[(a, init)].mean("member") - ens[(b, init)].mean("member"), "p": p}).load()
    missing = sorted({(c, i) for c in CASES for i in INIT_MONTHS if i in CASES[c]["init_months"]} - set(ens))
    print(f"{field}: {len(ens)} ensembles" + (f"; missing {missing}" if missing else ""))
    del ens

### Land-mean difference by lead

Solid: ensemble-mean difference (first minus second case); shading: 95% interval from the member land means.

In [ ]:
def pair_label(pair):
    return f"{pair[0]} minus {pair[1]}"


PAIR_COLORS = {PAIRS[0]: "tab:purple", PAIRS[1]: "tab:blue", PAIRS[2]: "tab:orange"}

fig, axes = plt.subplots(len(FIELDS), len(INIT_MONTHS), figsize=(4.2 * len(INIT_MONTHS), 2.4 * len(FIELDS)), sharex=True, squeeze=False)
for row, field in enumerate(FIELDS):
    for col, init in enumerate(INIT_MONTHS):
        ax = axes[row, col]
        for pair in PAIRS:
            ka, kb = (pair[0], init, field), (pair[1], init, field)
            if ka not in land_mean or kb not in land_mean:
                continue
            xa, xb = land_mean[ka], land_mean[kb]
            diff = xa.mean("member") - xb.mean("member")
            se = np.sqrt(xa.var("member") / xa.sizes["member"] + xb.var("member") / xb.sizes["member"])
            lead = diff["lead"].values
            ax.fill_between(lead, diff - 2 * se, diff + 2 * se, color=PAIR_COLORS[pair], alpha=0.2)
            ax.plot(lead, diff, color=PAIR_COLORS[pair], lw=2, label=pair_label(pair))
        ax.axhline(0.0, color="0.5", lw=0.8)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(f"{MONTH_NAMES[init]} {CASE_YEAR} start")
        if col == 0:
            ax.set_ylabel(f"{field}\n({rc.FIELDS[field]['units']})")
        if row == len(FIELDS) - 1:
            ax.set_xlabel("lead (months)")
handles = {l: h for ax in axes.flat for h, l in zip(*ax.get_legend_handles_labels())}
fig.legend(handles.values(), handles.keys(), loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Land-mean IC sensitivity", fontsize=13, y=1.03)
fig.tight_layout()
print(rc.save_figure(fig, "rtenso2026", "ic_sensitivity", "land_mean_difference"))
plt.show()

### IC memory: fraction of land with a significant difference

In [ ]:
fig, axes = plt.subplots(1, len(PAIRS), figsize=(5.5 * len(PAIRS), 4), sharey=True, squeeze=False)
styles = dict(zip(INIT_MONTHS, ["-", "--", "-.", ":"]))
colors = dict(zip(FIELDS, plt.cm.tab10.colors))
for ax, pair in zip(axes[0], PAIRS):
    for field in FIELDS:
        for init in INIT_MONTHS:
            frac = sig_fraction.get((pair, init, field))
            if frac is None:
                continue
            ax.plot(frac["lead"], frac, color=colors[field], ls=styles[init], lw=1.6,
                    label=field if init == INIT_MONTHS[0] or (pair, INIT_MONTHS[0], field) not in sig_fraction else None)
    ax.axhline(ALPHA, color="k", lw=0.8, ls=":")
    ax.set_title(pair_label(pair), fontsize=10)
    ax.set_xlabel("lead (months)")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel(f"land fraction with p < {ALPHA}")
axes[0, 0].legend(fontsize=8, title="line style = start month", title_fontsize=8)
fig.suptitle("How long the land/sea-ice/river IC difference persists (dotted: chance level)", fontsize=12)
fig.tight_layout()
print(rc.save_figure(fig, "rtenso2026", "ic_sensitivity", "significant_land_fraction"))
plt.show()

### Difference maps for `MAP_FIELD`

Stippling marks p < 0.05.

In [ ]:
for pair in PAIRS:
    inits = [i for i in INIT_MONTHS if (pair, i) in maps]
    if not inits:
        print(f"{pair_label(pair)}: no {MAP_FIELD} data")
        continue
    vmax = float(np.nanpercentile(np.abs(np.concatenate([maps[(pair, i)]["diff"].values.ravel() for i in inits])), 98))
    fig, axes = plt.subplots(len(MAP_LEADS), len(inits), figsize=(4.8 * len(inits), 2.7 * len(MAP_LEADS)),
                             subplot_kw={"projection": ccrs.Robinson(central_longitude=180)}, squeeze=False)
    for col, init in enumerate(inits):
        ds = maps[(pair, init)]
        for row, lead in enumerate(MAP_LEADS):
            ax = axes[row, col]
            d, p = ds["diff"].sel(lead=lead), ds["p"].sel(lead=lead)
            mesh = ax.pcolormesh(d["lon"], d["lat"], d, cmap="BrBG" if MAP_FIELD != "TSA" else "RdBu_r",
                                 vmin=-vmax, vmax=vmax, transform=ccrs.PlateCarree(), rasterized=True)
            sig = p.where(p < ALPHA)
            yy, xx = np.meshgrid(sig["lat"], sig["lon"], indexing="ij")
            mask = np.isfinite(sig.values)
            ax.scatter(xx[mask][::9], yy[mask][::9], s=0.3, color="k", transform=ccrs.PlateCarree())
            ax.coastlines(lw=0.4)
            ax.set_global()
            t = ds["time"].values[lead] if "time" in ds.coords else None
            ax.set_title(f"{MONTH_NAMES[init]} start, lead {lead}" + (f" ({t.year}-{t.month:02d})" if t is not None else ""), fontsize=9)
    fig.colorbar(mesh, ax=axes, orientation="horizontal", fraction=0.03, pad=0.04, shrink=0.6,
                 label=f"{MAP_FIELD} ({rc.FIELDS[MAP_FIELD]['units']})")
    fig.suptitle(f"{MAP_FIELD}: {pair_label(pair)}", fontsize=12)
    print(rc.save_figure(fig, "rtenso2026", "ic_sensitivity", MAP_FIELD, pair[0], "minus", pair[1]))
    plt.show()

### Sea-ice area against HadISST2

Hemispheric sea-ice area from `timeMonthly_avg_iceAreaCell` (members thin, ensemble mean thick).

In [ ]:
%%time
obs_sic = rc.obs_monthly("SICE", mode=derivation_mode)
obs_area = rc.hemispheric_area(obs_sic.sel(time=slice(f"{CASE_YEAR}-01", f"{CASE_YEAR + 2}-12")))

fig, axes = plt.subplots(2, len(INIT_MONTHS), figsize=(4.2 * len(INIT_MONTHS), 6.5), sharex="col", squeeze=False)
any_model = False
for col, init in enumerate(INIT_MONTHS):
    for row, hemi in enumerate(["NH", "SH"]):
        ax = axes[row, col]
        ax.plot(to_datetime(obs_area["time"].values), obs_area.sel(hemisphere=hemi), color="k", lw=2.2, label="HadISST2")
        for case in CASES:
            if init not in CASES[case]["init_months"]:
                continue
            ice = rc.open_hindcast(CASES[case]["case_prefix"], CASE_YEAR, init, "SICE")
            if ice is None:
                continue
            any_model = True
            area = rc.hemispheric_area(ice).sel(hemisphere=hemi)
            x, color = to_datetime(ice["time"].values), CASES[case]["color"]
            for m in area["member"].values:
                ax.plot(x, area.sel(member=m), color=color, lw=0.5, alpha=0.4)
            ax.plot(x, area.mean("member"), color=color, lw=2, label=case)
        ax.set_xlim(datetime(CASE_YEAR, init, 1), datetime(CASE_YEAR + 2, init, 1))
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(f"{MONTH_NAMES[init]} {CASE_YEAR} start")
        if col == 0:
            ax.set_ylabel(f"{hemi} sea-ice area (10^6 km^2)")
        ax.tick_params(axis="x", labelrotation=45)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
if not any_model:
    print("No sea-ice hindcasts post-processed yet (run ice_mon1); showing HadISST2 only.")
print(rc.save_figure(fig, "rtenso2026", "ic_sensitivity", "sea_ice_area"))
plt.show()